In [1]:
import pandas as pd
import numpy as np
import pyarrow
import statsmodels
import sklearn
import openai

print("basic packages ok")

basic packages ok


In [3]:
import wrds
import litellm

print("wrds and litellm ok")

wrds and litellm ok


In [6]:
import os
from pathlib import Path

# Set working directory to workspace root (3 levels up from notebook)
notebook_dir = Path('/data/disk4/workspace/projects/union_glassdoor/notebooks')
workspace_root = notebook_dir.parent.parent.parent
os.chdir(workspace_root)
print(f"Working directory: {os.getcwd()}")

# Build union_dummy: match gvkeys in glassdoor mapping with union election data
# 1 if gvkey appears in union data, 0 otherwise

# Read the data
mapping_df = pd.read_csv('projects/glassdoor/data/company_mapping_clean.csv')
union_df = pd.read_csv('projects/union/outputs/union_election_rc_votes_matched_combined.csv')

print(f"Mapping rows: {len(mapping_df)}, non-null gvkey: {mapping_df['gvkey'].notna().sum()}")
print(f"Union rows: {len(union_df)}")

# Extract and normalize gvkeys from union data
def normalize_gvkey(x):
    """Normalize gvkey: strip spaces, remove NA-like values, remove trailing .0"""
    if pd.isna(x):
        return None
    x = str(x).strip()
    if x.lower() in ['', 'na', 'nan', 'none']:
        return None
    if x.endswith('.0') and x[:-2].isdigit():
        x = x[:-2]
    return x

# Build union gvkey set from matched_gvkey and gvkey_final
union_gvkeys = set()
for col in ['matched_gvkey', 'gvkey_final']:
    if col in union_df.columns:
        gvkeys = union_df[col].apply(normalize_gvkey).dropna()
        union_gvkeys.update(gvkeys)

print(f"Unique gvkeys in union data: {len(union_gvkeys)}")

# Create dummy variable in mapping data
mapping_df['union_dummy'] = mapping_df['gvkey'].apply(
    lambda x: 1 if normalize_gvkey(x) in union_gvkeys else 0
)

print(f"\nResults:")
print(f"union_dummy = 1: {(mapping_df['union_dummy'] == 1).sum()}")
print(f"union_dummy = 0: {(mapping_df['union_dummy'] == 0).sum()}")

Working directory: /data/disk4/workspace
Mapping rows: 14747, non-null gvkey: 8447
Union rows: 26523
Unique gvkeys in union data: 10099

Results:
union_dummy = 1: 2364
union_dummy = 0: 12383


/tmp/ipykernel_39489/3401906499.py:15: DtypeWarning: Columns (41) have mixed types. Specify dtype option on import or set low_memory=False.
  union_df = pd.read_csv('projects/union/outputs/union_election_rc_votes_matched_combined.csv')


In [8]:
mapping_df.head()

,employer_id,company_name,gvkey,compustat_name,match_method,glassdoor_url,union_dummy
0,100,Black Hills Corporation,2259.0,BLACK HILLS CORP,direct_name,https://www.glassdoor.com/Reviews/Black-Hills-...,0
1,1000,SJW,9324.0,SJW GROUP,russell3000_supplement,https://www.glassdoor.com/Reviews/SJW-Reviews-...,1
2,10004,Caliper Life Sciences,127455.0,CALIPER LIFE SCIENCES INC,direct_name,https://www.glassdoor.com/Reviews/Caliper-Life...,0
3,10007,Taco John s International,6270.0,JOHNSON INTERNATIONAL CORP,public_info,https://www.glassdoor.com/Reviews/Taco-John-s-...,0
4,10008,HealthStream,133766.0,HEALTHSTREAM INC,direct_name,https://www.glassdoor.com/Reviews/HealthStream...,0


In [13]:
mapping_df[mapping_df['union_dummy'] == 1].iloc[0]['glassdoor_url']

'https://www.glassdoor.com/Reviews/SJW-Reviews-E1000.htm'

In [14]:
# Save and verify results
print("\n" + "="*60)
print("SAVING AND VERIFYING UNION DUMMY")
print("="*60)

# Save to file
output_path = 'projects/glassdoor/data/company_mapping_clean_with_union_dummy.csv'
mapping_df.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")

# Verify the saved file
verification_df = pd.read_csv(output_path)
print(f"\nVerification:")
print(f"File rows: {len(verification_df)}")
print(f"Columns: {list(verification_df.columns)}")
print(f"union_dummy = 1: {(verification_df['union_dummy'] == 1).sum()}")
print(f"union_dummy = 0: {(verification_df['union_dummy'] == 0).sum()}")

# Show sample
print("\nFirst 5 rows with union_dummy:")
print(verification_df[['employer_id', 'company_name', 'gvkey', 'union_dummy']].head())

print("\nSample of union_dummy=1 (companies in both datasets):")
print(verification_df[verification_df['union_dummy'] == 1][['employer_id', 'company_name', 'gvkey']].head(10))


SAVING AND VERIFYING UNION DUMMY

Saved to projects/glassdoor/data/company_mapping_clean_with_union_dummy.csv

Verification:
File rows: 14747
Columns: ['employer_id', 'company_name', 'gvkey', 'compustat_name', 'match_method', 'glassdoor_url', 'union_dummy']
union_dummy = 1: 2364
union_dummy = 0: 12383

First 5 rows with union_dummy:
   employer_id               company_name     gvkey  union_dummy
0          100    Black Hills Corporation    2259.0            0
1         1000                        SJW    9324.0            1
2        10004      Caliper Life Sciences  127455.0            0
3        10007  Taco John s International    6270.0            0
4        10008               HealthStream  133766.0            0

Sample of union_dummy=1 (companies in both datasets):
    employer_id          company_name     gvkey
1          1000                   SJW    9324.0
6         10018  Office Star Products   31012.0
8         10021          OPI Products    8573.0
11        10034   Conbraco 